# Module 2: Prompt Templates & Output Parsers

In this notebook, we will explore:
1. **Prompt Templates**: Constructing reusable prompts for single variables and chat formats.
2. **Basic Parsers**: Converting comma-separated outputs into standard Python lists.
3. **Structured Parsers**: Instructing the model to return valid JSON, and parsing it into Python dictionaries.
4. **Pydantic Validation**: Injecting schema schemas and converting responses into validated Pydantic objects.

### Step 1: Initialize model connection

In [26]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.0, # 0.0 is best for structured extraction and formatting
)
print("Model client connected!")

Model client connected!


---
## 1. Prompt Templates

Let's see how `PromptTemplate` and `ChatPromptTemplate` generate and format prompts.

In [27]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# A. Basic Text Prompt
prompt_template = PromptTemplate.from_template(
    "Write a technical explanation of what {technology} is, suitable for a {audience}."
)
print("--- Standard PromptTemplate ---")
print(prompt_template.format(technology="Docker", audience="high school student"))

# B. Chat-focused Prompt
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional chef. You respond only in the style of {cooking_style}."),
    ("human", "Recommend a dinner recipe using: {ingredients}")
])
print("\n--- ChatPromptTemplate Messages ---")
formatted_messages = chat_template.format_messages(
    cooking_style="Gordon Ramsay (angry and passionate)",
    ingredients="eggs, tomatoes, and old bread"
)
for msg in formatted_messages:
    print(f"[{type(msg).__name__}]: {msg.content}")

--- Standard PromptTemplate ---
Write a technical explanation of what Docker is, suitable for a high school student.

--- ChatPromptTemplate Messages ---
[SystemMessage]: You are a professional chef. You respond only in the style of Gordon Ramsay (angry and passionate).
[HumanMessage]: Recommend a dinner recipe using: eggs, tomatoes, and old bread


---
## 2. Basic Output Parser: Comma-Separated Lists

Let's write a prompt that asks for a list, and convert the resulting comma-separated string into a Python list of elements.

In [28]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()
format_instructions = list_parser.get_format_instructions()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items exactly as instructed.\n{instructions}"),
    ("human", "List 5 programming languages for data science.")
])

# Run invocation
formatted = list_prompt.format_messages(instructions=format_instructions)
raw_response = model.invoke(formatted)
print("Raw LLM Output:\n", repr(raw_response.content))

# Parse response
parsed_list = list_parser.parse(raw_response.content)
print("\nParsed Python List:\n", parsed_list)
print("Type:", type(parsed_list))

Raw LLM Output:
 'Python, R, Julia, SAS, Scala'

Parsed Python List:
 ['Python', 'R', 'Julia', 'SAS', 'Scala']
Type: <class 'list'>


---
## 3. Structured JSON Parsing (`JsonOutputParser`)

To extract structured values (dictionaries) without strict schema matching, we use the `JsonOutputParser`.

In [29]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_instructions = json_parser.get_format_instructions()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a database extractor. Return information in valid JSON only.\n{instructions}"),
    ("human", "Create a profile for the historical figure: Julius Caesar.")
])

formatted_prompt = json_prompt.format_messages(instructions=json_instructions)
raw_json_res = model.invoke(formatted_prompt)
print("Raw JSON Response:\n", raw_json_res.content)

parsed_json = json_parser.parse(raw_json_res.content)
print("\nParsed Dictionary:")
print(parsed_json)
print("Type:", type(parsed_json))

Raw JSON Response:
 {
  "name": "Julius Caesar",
  "birth_date": "100 BC",
  "death_date": "15 March 44 BC",
  "birth_place": "Alba Longa, Roman Republic",
  "profession": "Roman general, statesman, and author",
  "key_roles": [
    "Military commander",
    "Consul",
    "Dictator for life"
  ],
  "notable_achievements": [
    "Conquered Gaul (58–50 BC)",
    "Crossed the Rubicon River, igniting civil war",
    "Implemented significant political and social reforms",
    "Wrote historical works, notably 'Commentarii de Bello Gallico'"
  ],
  "assassination": "Murdered on the Ides of March (15 March 44 BC) by a group of Roman senators led by Brutus and Cassius"
}

Parsed Dictionary:
{'name': 'Julius Caesar', 'birth_date': '100 BC', 'death_date': '15 March 44 BC', 'birth_place': 'Alba Longa, Roman Republic', 'profession': 'Roman general, statesman, and author', 'key_roles': ['Military commander', 'Consul', 'Dictator for life'], 'notable_achievements': ['Conquered Gaul (58–50 BC)', 'Cross

---
## 4. Advanced Pydantic Output Validation

Let's see how to define a structured output format using `Pydantic` models and enforce it using `PydanticOutputParser`.

In [30]:
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.output_parsers import PydanticOutputParser

# 1. Define the desired schema using Pydantic
class CharacterProfile(BaseModel):
    name: str = Field(description="The full name of the character")
    alignment: str = Field(description="Moral alignment (e.g. Good, Evil, Neutral)")
    abilities: List[str] = Field(description="List of key special abilities or powers")
    catchphrase: Optional[str] = Field(None, description="Memorable catchphrase, if any")
    power_level: int = Field(description="Numeric value representing power between 1 and 100")

# 2. Initialize the parser passing the schema
pydantic_parser = PydanticOutputParser(pydantic_object=CharacterProfile)
pydantic_instructions = pydantic_parser.get_format_instructions()

# Let's print the instructions to see how LangChain communicates the schema to the LLM:
print("--- AUTO-GENERATED INSTRUCTIONS ---")
print(pydantic_instructions)
print("-----------------------------------")

--- AUTO-GENERATED INSTRUCTIONS ---
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "The full name of the character", "title": "Name", "type": "string"}, "alignment": {"description": "Moral alignment (e.g. Good, Evil, Neutral)", "title": "Alignment", "type": "string"}, "abilities": {"description": "List of key special abilities or powers", "items": {"type": "string"}, "title": "Abilities", "type": "array"}, "catchphrase": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "description": "Memorable catchphrase, if any", "title": "Catchp

Now we pass the formatted instructions to the prompt, invoke the model, and parse the output into our `CharacterProfile` object.

In [31]:
pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative writer. Return structured data according to the format instructions.\n{instructions}"),
    ("human", "Create a profile for the superhero: Spider-Man.")
])

formatted = pydantic_prompt.format_messages(instructions=pydantic_instructions)
raw_pydantic_res = model.invoke(formatted)
print("Raw Output:\n", raw_pydantic_res.content)

character = pydantic_parser.parse(raw_pydantic_res.content)

print("\n--- PARSED PYDANTIC OBJECT ---")
print("Name:", character.name)
print("Alignment:", character.alignment)
print("Abilities:", character.abilities)
print("Catchphrase:", character.catchphrase)
print("Power Level:", character.power_level)
print("Type:", type(character))

Raw Output:
 {
  "name": "Peter Parker",
  "alignment": "Good",
  "abilities": [
    "Superhuman strength",
    "Enhanced agility",
    "Wall-crawling",
    "Spider-sense",
    "Expert web-shooter"
  ],
  "catchphrase": "With great power comes great responsibility.",
  "power_level": 75
}

--- PARSED PYDANTIC OBJECT ---
Name: Peter Parker
Alignment: Good
Abilities: ['Superhuman strength', 'Enhanced agility', 'Wall-crawling', 'Spider-sense', 'Expert web-shooter']
Catchphrase: With great power comes great responsibility.
Power Level: 75
Type: <class '__main__.CharacterProfile'>
